# Moxon Tuning Protocol

Use this notebook while iteratively tuning the antenna with a NanoVNA, RigExpert Stick, or similar analyzer. The repository pre-commit/pre-push hook clears notebook cell outputs, so this notebook writes the persistent record to `data/moxon_tuning_log.csv`.

Recommended workflow:

1. Measure the antenna using the process in `../TUNING.md`.
2. Enter the as-built geometry and analyzer readings in the `entry` dictionary.
3. Run `append_entry(update_front_to_back(entry))` after every physical change.
4. Commit the notebook plus the CSV log if the measurements should be part of the project history.
5. If a measurement is only a private field note, keep that CSV change uncommitted or copy it to a separate local file.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

LOG_PATH = Path('data/moxon_tuning_log.csv')
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    'timestamp_utc',
    'step',
    'antenna_name',
    'target_frequency_mhz',
    'wire_type',
    'wire_diameter_mm',
    'a_span_mm',
    'b_tail_left_mm',
    'b_tail_right_mm',
    'c_gap_left_mm',
    'c_gap_right_mm',
    'd_tail_left_mm',
    'd_tail_right_mm',
    'e_spacing_mm',
    'feed_gap_mm',
    'outer_pipe_mm',
    'center_pipe_mm',
    'analyzer',
    'calibration_plane',
    'measurement_height_m',
    'orientation',
    'choke_description',
    'coax_route',
    'f_min_swr_mhz',
    'swr_min',
    'target_swr',
    'r_ohm_at_target',
    'x_ohm_at_target',
    'return_loss_db_at_target',
    'front_db',
    'rear_db',
    'front_to_back_db',
    'change_made',
    'notes',
]

def load_log():
    if LOG_PATH.exists():
        return pd.read_csv(LOG_PATH)
    return pd.DataFrame(columns=COLUMNS)

def save_log(df):
    df = df.reindex(columns=COLUMNS)
    df.to_csv(LOG_PATH, index=False)
    return df

def append_entry(entry):
    row = {column: entry.get(column, '') for column in COLUMNS}
    row['timestamp_utc'] = entry.get('timestamp_utc') or datetime.now(timezone.utc).isoformat(timespec='seconds')
    df = load_log()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return save_log(df)

def update_front_to_back(entry):
    front = entry.get('front_db')
    rear = entry.get('rear_db')
    if front not in (None, '') and rear not in (None, ''):
        entry['front_to_back_db'] = float(front) - float(rear)
    return entry


## Add A Measurement

Edit the dictionary, run the cell, then run `append_entry(update_front_to_back(entry))`. Keep dimensions in millimeters and frequencies in MHz. Use blank strings for values that were not measured.


In [ ]:
entry = {
    'step': 0,
    'antenna_name': '2m Moxon prototype',
    'target_frequency_mhz': 145.0,
    'wire_type': 'insulated copper',
    'wire_diameter_mm': '',
    'a_span_mm': '',
    'b_tail_left_mm': '',
    'b_tail_right_mm': '',
    'c_gap_left_mm': '',
    'c_gap_right_mm': '',
    'd_tail_left_mm': '',
    'd_tail_right_mm': '',
    'e_spacing_mm': '',
    'feed_gap_mm': '',
    'outer_pipe_mm': '',
    'center_pipe_mm': '',
    'analyzer': 'NanoVNA',
    'calibration_plane': 'end of analyzer jumper at feed point',
    'measurement_height_m': '',
    'orientation': 'horizontal',
    'choke_description': '',
    'coax_route': 'away from feed point at right angle',
    'f_min_swr_mhz': '',
    'swr_min': '',
    'target_swr': '',
    'r_ohm_at_target': '',
    'x_ohm_at_target': '',
    'return_loss_db_at_target': '',
    'front_db': '',
    'rear_db': '',
    'change_made': 'initial build',
    'notes': '',
}


In [ ]:
# Uncomment after filling the entry dictionary with a real measurement.
# append_entry(update_front_to_back(entry))


## Review The Log

These cells regenerate summaries from the CSV. Their displayed output is intentionally not persistent in git because the hook clears notebook outputs before commit and push.


In [ ]:
log = load_log()
log.tail(10)


In [ ]:
numeric_columns = ['step', 'f_min_swr_mhz', 'swr_min', 'target_swr', 'r_ohm_at_target', 'x_ohm_at_target', 'front_to_back_db']
plot_df = load_log().copy()
for column in numeric_columns:
    plot_df[column] = pd.to_numeric(plot_df[column], errors='coerce')

plot_df.plot(x='step', y=['f_min_swr_mhz', 'swr_min', 'target_swr', 'front_to_back_db'], marker='o', subplots=True, figsize=(9, 8))
